# SC17 - Classification Backbone Benchmark
Compares every trained backbone on the **same held-out test split**, item-level (views aggregated),
so it is apples-to-apples. Includes the **vision transformer (DINOv2 ViT-S)** alongside the CNNs and
the fusion model. Produces (1) a comparison table + one headline comparison bar, and (2) for **each
model** its own diagnostics: training curves, confusion matrices, ROC curves, per-class material F1,
reliability - one model per PNG.

**Attach:** `thesis-common-10c` + the Results dataset(s) holding `logits_*/history_*/metrics_*`.
No GPU needed (reads cached logits); attach GPU only if you want to re-evaluate weights.

In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)

# generic finders across kaggle inputs + local Results
ROOTS = ['/kaggle/input', 'Results', '.']
def find(fn):
    for r in ROOTS:
        h = sorted(glob.glob(f'{r}/**/{fn}', recursive=True), key=len)
        if h: return h[0]
    return None
def find_all(pat):
    seen = {}
    for r in ROOTS:
        for p in sorted(glob.glob(f'{r}/**/{pat}', recursive=True), key=len):
            seen.setdefault(os.path.basename(p), p)
    return list(seen.values())
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import pandas as pd, torch

CLS_DIR = os.path.join(OUT, 'clsfig'); os.makedirs(CLS_DIR, exist_ok=True)

# model family so the transformer is explicit in the table
FAMILY = {
 'efficientnet_b0':'CNN', 'mobilenetv3_large_100':'CNN', 'resnet50':'CNN',
 'convnext_tiny':'CNN (ConvNet)', 'convnext_small.fb_in22k_ft_in1k':'CNN (ConvNet)',
 'vit_small_patch14_dinov2.lvd142m':'Transformer (ViT)', 'fusion':'Ensemble',
}
PRETTY = {
 'efficientnet_b0':'EfficientNet-B0', 'mobilenetv3_large_100':'MobileNetV3-L',
 'resnet50':'ResNet-50', 'convnext_tiny':'ConvNeXt-T',
 'convnext_small.fb_in22k_ft_in1k':'ConvNeXt-S (ours)',
 'vit_small_patch14_dinov2.lvd142m':'DINOv2 ViT-S', 'fusion':'Fusion (3-bb)',
}
def short(nm): return PRETTY.get(nm, nm)

def load_logits(path):
    d = np.load(path, allow_pickle=True); it = d['test_items']
    logits = {k: d[f'test_log_{k}'] for k in ['material','condition','route','threat']}
    ys     = {k: d[f'test_y_{k}']  for k in ['material','condition','route','threat']}
    name = os.path.basename(path).replace('logits_','').replace('.npz','')
    return name, logits, ys, it

LOGITS = find_all('logits_*.npz')
print('found', len(LOGITS), 'logit files:')
for p in LOGITS: print('  ', os.path.basename(p))

In [ ]:
# ---- comparison table from full_report on each model's logits (item-level) ----
recs = {}
for p in LOGITS:
    name, logits, ys, it = load_logits(p)
    rep = T.full_report(logits, ys, it)
    s = rep['scalars']
    recs[name] = {
        'family': FAMILY.get(name,'CNN'),
        'material_acc': s['material']['acc'], 'material_f1': s['material']['macro_f1'],
        'condition_auroc': s['condition']['auroc'], 'route_f1': s['route']['macro_f1'],
        'threat_qwk': s['threat']['qwk'], 'threat_mae': s['threat']['mae'],
        'material_ece': s['material']['ece'],
    }
comp = pd.DataFrame(recs).T
metric_cols = ['material_acc','material_f1','condition_auroc','route_f1','threat_qwk','threat_mae','material_ece']
for c in metric_cols: comp[c] = comp[c].astype(float)
comp = comp.sort_values('material_f1', ascending=False)
comp.insert(0,'model',[short(i) for i in comp.index])
comp.to_csv(os.path.join(OUT,'classification_comparison.csv'))
json.dump(comp.reset_index().rename(columns={'index':'backbone'}).to_dict('records'),
          open(os.path.join(OUT,'classification_comparison.json'),'w'), indent=2)
print(comp[['model','family']+metric_cols].round(3).to_string())

In [ ]:
# ---- LaTeX table (bold = best per column; higher is better except MAE/ECE) ----
show = ['material_acc','material_f1','condition_auroc','route_f1','threat_qwk','threat_mae','material_ece']
head = ['Mat.\\,Acc','Mat.\\,F1','Cond.\\,AUROC','Route\\,F1','Threat\\,QWK','Threat\\,MAE','Mat.\\,ECE']
lower_better = {'threat_mae','material_ece'}
best = {c:(comp[c].idxmin() if c in lower_better else comp[c].idxmax()) for c in show}
lines = [r'\begin{table}[t]', r'\centering',
         r'\caption{Classification backbone comparison on the held-out test split (item-level, views aggregated). Best per column in bold.}',
         r'\label{tab:cls_compare}',
         r'\begin{tabular}{ll'+'c'*len(show)+'}', r'\toprule',
         r'Backbone & Family & '+' & '.join(head)+r' \\', r'\midrule']
for idx,row in comp.iterrows():
    cells=[]
    for c in show:
        v=f"{row[c]:.3f}"
        if idx==best[c]: v=r'\textbf{'+v+'}'
        cells.append(v)
    nm = short(idx).replace('_','\\_'); fam = row['family']
    lines.append(f'{nm} & {fam} & '+' & '.join(cells)+r' \\')
lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
open(os.path.join(OUT,'classification_comparison.tex'),'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ---- headline comparison bar (this ONE figure is intentionally multi-model: the comparison) ----
bars = [('material_f1','Material macro-F1'),('threat_qwk','Threat QWK'),
        ('route_f1','Route macro-F1'),('condition_auroc','Condition AUROC')]
for col,title in bars:
    s = comp[col].astype(float).sort_values()
    colors = ['#E45756' if i=='convnext_small.fb_in22k_ft_in1k'
              else ('#54A24B' if 'vit' in i else '#4C78A8') for i in s.index]
    plt.figure(figsize=(6.4,3.6))
    plt.barh([short(i) for i in s.index], s.values, color=colors)
    for y,v in enumerate(s.values): plt.text(v+0.005, y, f'{v:.3f}', va='center', fontsize=8)
    plt.xlim(0, 1.05); plt.title(f'Backbone comparison - {title}')
    plt.tight_layout(); plt.savefig(os.path.join(OUT,f'cls_compare_{col}.png'), dpi=200); plt.close()
print('saved 4 comparison bars: cls_compare_*.png')

In [ ]:
# ============ PER-MODEL diagnostics (one model per PNG) ============
THREAT_LABELS = [str(i) for i in range(T.NUM_THREAT)]
def savecm(cm, labels, title, path):
    cm = np.array(cm, dtype=float)
    plt.figure(figsize=(4.6,4.2)); plt.imshow(cm, cmap='Blues')
    plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
    plt.yticks(range(len(labels)), labels, fontsize=7)
    thr = cm.max()/2 if cm.max() else 0.5
    for i in range(len(labels)):
        for j in range(len(labels)):
            if cm[i,j]: plt.text(j,i,int(cm[i,j]),ha='center',va='center',
                                 color='white' if cm[i,j]>thr else 'black', fontsize=7)
    plt.ylabel('true'); plt.xlabel('predicted'); plt.title(title, fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=200); plt.close()

HIST = {os.path.basename(p): p for p in find_all('history_*.csv')}
def hist_for(name):
    for k,p in HIST.items():
        if name in k or k.replace('history_','').replace('.csv','')==name: return p
    return None

for p in LOGITS:
    name, logits, ys, it = load_logits(p)
    d = os.path.join(CLS_DIR, name); os.makedirs(d, exist_ok=True)
    rep = T.full_report(logits, ys, it)
    agg, uniq, idx = T.aggregate_by_item(logits, it); lab = T.item_labels(ys, it, idx)
    tag = short(name)

    # (1) confusion matrices
    savecm(rep['confusion']['material'], T.MATERIALS, f'{tag} - Material confusion',
           os.path.join(d, f'{name}_confusion_material.png'))
    savecm(rep['confusion']['route'], T.ROUTES, f'{tag} - Route confusion',
           os.path.join(d, f'{name}_confusion_route.png'))
    savecm(rep['confusion']['threat'], THREAT_LABELS, f'{tag} - Threat confusion',
           os.path.join(d, f'{name}_confusion_threat.png'))

    # (2) condition ROC (binary)
    y = lab['condition'].astype(int)
    if len(np.unique(y))>1:
        prob = T.sigmoid_np(agg['condition']); fpr,tpr,_ = roc_curve(y,prob); a=auc(fpr,tpr)
        plt.figure(figsize=(4.6,4.4)); plt.plot(fpr,tpr,lw=2,color='#E45756',label=f'AUC={a:.3f}')
        plt.plot([0,1],[0,1],'k--',alpha=.4); plt.xlabel('FPR'); plt.ylabel('TPR')
        plt.title(f'{tag} - Condition ROC (Good vs Poor)'); plt.legend(loc='lower right')
        plt.tight_layout(); plt.savefig(os.path.join(d,f'{name}_condition_roc.png'),dpi=200); plt.close()

    # (3) route one-vs-rest ROC (macro)
    rp = T.softmax_np(agg['route']); yr = lab['route']
    plt.figure(figsize=(4.6,4.4))
    for ci,rn in enumerate(T.ROUTES):
        yb=(yr==ci).astype(int)
        if len(np.unique(yb))<2: continue
        fpr,tpr,_=roc_curve(yb,rp[:,ci]); plt.plot(fpr,tpr,lw=1.5,label=f'{rn} ({auc(fpr,tpr):.2f})')
    plt.plot([0,1],[0,1],'k--',alpha=.4); plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'{tag} - Route ROC (one-vs-rest)'); plt.legend(fontsize=7,loc='lower right')
    plt.tight_layout(); plt.savefig(os.path.join(d,f'{name}_route_roc.png'),dpi=200); plt.close()

    # (4) per-class material F1
    pc = rep['per_class']['material']
    items = sorted(pc.items(), key=lambda kv: kv[1]['f1'])
    names=[k for k,_ in items]; f1=[v['f1'] for _,v in items]; sup=[v['support'] for _,v in items]
    plt.figure(figsize=(6.2,4.0)); b=plt.barh(names,f1,color='#E45756')
    for bar,sv in zip(b,sup): plt.text(bar.get_width()+0.01,bar.get_y()+bar.get_height()/2,f'n={sv}',va='center',fontsize=7)
    plt.xlim(0,1.1); plt.xlabel('F1'); plt.title(f'{tag} - Material per-class F1')
    plt.tight_layout(); plt.savefig(os.path.join(d,f'{name}_material_perclass_f1.png'),dpi=200); plt.close()

    # (5) reliability (material confidence vs accuracy)
    mp = T.softmax_np(agg['material']); conf = mp.max(1); pred = mp.argmax(1); corr=(pred==lab['material']).astype(float)
    bins=np.linspace(0,1,11); ctr=(bins[:-1]+bins[1:])/2; accb=[]; cfb=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(conf>=lo)&(conf<hi)
        accb.append(corr[m].mean() if m.sum() else np.nan); cfb.append(conf[m].mean() if m.sum() else np.nan)
    plt.figure(figsize=(4.6,4.4)); plt.plot([0,1],[0,1],'k--',alpha=.4,label='perfect')
    plt.plot(ctr,accb,'o-',color='#4C78A8',label='model')
    plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.ylim(0,1); plt.xlim(0,1)
    plt.title(f'{tag} - Reliability (material)  ECE={rep["scalars"]["material"]["ece"]:.3f}')
    plt.legend(loc='upper left'); plt.tight_layout()
    plt.savefig(os.path.join(d,f'{name}_reliability_material.png'),dpi=200); plt.close()

    # (6) training curves from history (single model)
    hp = hist_for(name)
    if hp:
        h = pd.read_csv(hp)
        fig,ax=plt.subplots(1,2,figsize=(9,3.4))
        ax[0].plot(h.epoch,h.train_loss,'-o',ms=3,label='train')
        if 'val_loss' in h: ax[0].plot(h.epoch,h.val_loss,'-o',ms=3,label='val')
        ax[0].set_title(f'{tag} - loss'); ax[0].set_xlabel('epoch'); ax[0].legend(fontsize=8)
        for col,l in [('val_material_f1','matF1'),('val_route_f1','routeF1'),
                      ('val_threat_qwk','threatQWK'),('val_condition_f1','condF1')]:
            if col in h: ax[1].plot(h.epoch,h[col],'-o',ms=3,label=l)
        ax[1].set_title(f'{tag} - val metrics'); ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=8); ax[1].set_ylim(0,1)
        plt.tight_layout(); plt.savefig(os.path.join(d,f'{name}_training_curves.png'),dpi=200); plt.close()
    print('done:', tag, '->', d)
print('\nAll per-model diagnostics under clsfig/<model>/  (one model per PNG).')